In [1]:
import os
import sys
import yaml
from collections import namedtuple
import numpy as np
import configparser

home_path = "/home/fc-3auid-3af522edce-2ddb4d-2d4b8c-2d8500-2df5376270c3e0"
module_path = os.path.join(home_path, "gitprojects/gref4hsi")  # Replace with the actual path
sys.path.append(module_path)

from gref4hsi.utils.config_utils import prepend_data_dir_to_relative_paths, customize_config


# We begin by defining all the stuff that process_hyperspectral does such as the config file:
def define_processing_vars(config_yaml, specim_mission_folder, geoid_path, config_template_path, lab_calibration_path, fast_mode = False):
# Read flight-specific yaml file
    with open(config_yaml, 'r') as file:  
        config_data = yaml.safe_load(file)
    
    
    # assigning the arguments to variables for simple backwards compatibility
    SPECIM_MISSION_FOLDER = specim_mission_folder
    EPSG_CODE = config_data['mission_epsg']
    RESOLUTION_ORTHOMOSAIC = config_data['resolution_orthomosaic']
    CALIBRATION_DIRECTORY = lab_calibration_path
    
    
    dem_fold = os.path.join(specim_mission_folder, "dem")

    if not os.path.exists(dem_fold):
        print('DEM folder does not exist so Geoid is used as terrain instead')
        TERRAIN_TYPE = "geoid"
    else:
        if not os.listdir(dem_fold):
            #print(f"The folder '{dem_fold}' is empty so Geoid is used as terrain instead.")
            TERRAIN_TYPE = "geoid"
        else:
            # If there is a folder and it is not empty
            # Find the only file that is there
            files = [f for f in os.listdir(dem_fold) if f not in ('.', '..')]
            DEM_PATH = os.path.join(dem_fold, files[0])
            #print(f"The file '{DEM_PATH}' is used as terrain.")
            TERRAIN_TYPE = "dem_file"
    
    
    # Do coregistration if there is an orthomosaic to compare under "orthomosaic"
    #do_coreg = True
    ortho_ref_fold = os.path.join(specim_mission_folder, "orthomosaic")
    do_coreg = False
    if not os.path.exists(ortho_ref_fold):
        print('Coregistration is not done, as there was no reference orthomosaic')
        
    else:
        if not os.listdir(ortho_ref_fold):
            print('Coregistration is not done, as there was no reference orthomosaic')
        else:
            # If there is a folder and it is not empty
            # Find the only file that is there
            ortho_ref_file = [f for f in os.listdir(ortho_ref_fold) if f not in ('.', '..')][0]
            do_coreg = True
            print(f"The file '{ortho_ref_file}' is used as as reference orthomosaic.")
            
    
    
    GEOID_PATH = geoid_path

    # Settings associated with preprocessing of data from Specim Proprietary data to pipeline-compatible data
    SettingsPreprocess = namedtuple('SettingsPreprocessing', ['dtype_datacube', 
                                                                            'lines_per_chunk', 
                                                                            'specim_raw_mission_dir',
                                                                            'cal_dir',
                                                                            'reformatted_missions_dir',
                                                                            'rotation_matrix_hsi_to_body',
                                                                            'translation_body_to_hsi',
                                                                            'config_file_name'])

    config_specim_preprocess = SettingsPreprocess(dtype_datacube = np.float32, # The data type for the datacube
                                lines_per_chunk = 2000,  # Raw datacube is chunked into this many lines. GB_per_chunk = lines_per_chunk*n_pixels*n_bands*4 bytes
                                specim_raw_mission_dir = SPECIM_MISSION_FOLDER, # Folder containing several mission
                                cal_dir = CALIBRATION_DIRECTORY,  # Calibration directory holding all calibrations at all binning levels
                                reformatted_missions_dir = os.path.join(SPECIM_MISSION_FOLDER, 'processed'), # The fill value for empty cells (select values not occcuring in cube or ancillary data)
                                rotation_matrix_hsi_to_body = np.array([[0, 1, 0],
                                                                        [-1, 0, 0],
                                                                        [0, 0, 1]]), # Rotation matrix R rotating so that vec_body = R*vec_hsi.
                                translation_body_to_hsi = np.array([0, 0, 0]), # Translation t so that vec_body_to_object = vec_hsi_to_object + t
                                # For large files, RAM issues could be a concern. For rectified files exeeding this size, data is written chunk-wize to a memory map.
                                config_file_name = 'configuration.ini')



    # Where to place the config
    DATA_DIR = config_specim_preprocess.reformatted_missions_dir
    config_file_mission = os.path.join(DATA_DIR, 'configuration.ini')


    # Set the data directory for the mission, and create empty folder structure
    prepend_data_dir_to_relative_paths(config_path=config_template_path, DATA_DIR=DATA_DIR)

    # Non-default settings
    custom_config = {'General':
                        {'mission_dir': DATA_DIR,
                        'model_export_type': TERRAIN_TYPE, # Ray trace onto geoid
                        'max_ray_length': 150}, # Max distance in meters from spectral imager to seafloor. Specim does not fly higher

                    'Coordinate Reference Systems':
                        {'proj_epsg' : EPSG_CODE, # The projected CRS UTM 32, common on mainland norway
                        'geocsc_epsg_export' : 4978, # 3D cartesian system for earth consistent with GPS frame (but inconsistent with eurasian techtonic plate)
                        'dem_epsg' : EPSG_CODE, # (Optional) If you have a DEM this can be used
                        'pos_epsg_orig' : 4978}, # The CRS of the positioning data we deliver to the georeferencing

                    'Orthorectification':
                        {'resample_rgb_only': False, # True can be good choice for speed during DEV
                         'resample_ancillary': True,
                        'resolutionhyperspectralmosaic': RESOLUTION_ORTHOMOSAIC, # Resolution in m
                        'raster_transform_method': 'north_east'}, # North-east oriented rasters.
                    
                    'HDF.raw_nav': {
                        'rotation_reference_type' : 'eul_ZYX', # The vehicle orientations are given in Yaw, Pitch, Roll from the NAV system
                        'is_global_rot' : False, # The vehicles orientations from NAV system are Yaw, Pitch, Roll
                        'eul_is_degrees' : True}, # And given in degrees
                    'Absolute Paths': {
                        'geoid_path' : GEOID_PATH,
                        'orthomosaic_reference_folder' : os.path.join(specim_mission_folder, "orthomosaic"),
                        'ref_ortho_reshaped' : os.path.join(DATA_DIR, "Intermediate", "RefOrthoResampled"),
                        'ref_gcp_path' : os.path.join(DATA_DIR, "Intermediate", "gcp.csv"),
                        'calib_file_coreg' : os.path.join(DATA_DIR, "Output", "HSI_coreg.xml"),
                        # (above) The georeferencing allows processing using norwegian geoid NN2000 and worldwide EGM2008. Also, use of seafloor terrain models are supported. '
                        # At the moment refractive ray tracing is not implemented, but it could be relatively easy by first ray tracing with geoid+tide, 
                        # and then ray tracing from water
                        #'tide_path' : 'D:/HyperspectralDataAll/HI/2022-08-31-060000-Remoy-Specim/Input/tidevann_nn2000_NMA.txt'
                        },
                    
                    # If coregistration is done, then the data must be stored after processing somewhere
                    'HDF.coregistration': {
                            'position_ecef': 'processed/coreg/position_ecef',
                            'quaternion_ecef' : 'processed/coreg/quaternion_ecef'
                        },
                    # These are the ancillary layers to be orthorectified (can select from all entities in "Georeferencing")
                    'Ancillary': {
                            'position_ecef' : 'processed/nav/position_hsi_ecef',
                            'quaternion_ecef' : 'processed/nav/quaternion_hsi_ecef',
                            'points_ecef_crs' : 'processed/georef/points_ecef_crs',
                            #'points_hsi_crs' : 'processed/georef/point_hsi_frame',
                            #'normals_hsi_crs' : 'processed/georef/normals_hsi_frame',
                            'theta_v' : 'processed/georef/theta_v',
                            'theta_s' : 'processed/georef/theta_s',
                            'phi_v' : 'processed/georef/phi_v',
                            'phi_s' : 'processed/georef/phi_s',
                            #'normals_ned_crs' : 'processed/georef/normals_ned_crs',
                            'unix_time_grid' : 'processed/georef/unix_time_grid', # h5 path
                            'pixel_nr_grid': 'processed/georef/pixel_nr_grid', # h5 path
                            #'frame_nr_grid' : 'processed/georef/frame_nr_grid',
                            #'hsi_tide_gridded' : 'processed/georef/hsi_tide_gridded',
                            'hsi_alts_msl' : 'processed/georef/hsi_alts_msl'
                        }
                    
    }

    if TERRAIN_TYPE == 'geoid':
        custom_config['Absolute Paths']['geoid_path'] = GEOID_PATH
        #'geoid_path' : 'data/world/geoids/egm08_25.gtx'
    elif TERRAIN_TYPE == 'dem_file':
        custom_config['Absolute Paths']['dem_path'] = DEM_PATH

    
    
    if do_coreg:
        # No need to orthorectify the data cube initially when coregistration with RGB composites is done
        custom_config['Orthorectification']['resample_rgb_only'] = True
        
        # Here you can set which camera parameters to optimize
        cam_calibrate_dict = {'calibrate_boresight': False,
                          'calibrate_camera': False,
                          'calibrate_lever_arm': False,
                          'calibrate_cx': False,
                          'calibrate_f': False,
                          'calibrate_k1': False,
                          'calibrate_k2': False,
                          'calibrate_k3': False
                          }

        # Here you can set which time-varying errors to estimate
        calibrate_dict_extr = {'calibrate_pos_x': True,
                          'calibrate_pos_y': True,
                          'calibrate_pos_z': True,
                          'calibrate_roll': False,
                          'calibrate_pitch': False,
                          'calibrate_yaw': True}
        
        coreg_dict = {'calibrate_dict': cam_calibrate_dict,
                      'calibrate_per_transect': True, # Whether to calibrate on each transect seperately (True) or to use an entire set of transects for calibration (False)
                      'calibrate_dict_extr': calibrate_dict_extr,
                      'time_node_spacing': 10, #s (set to really large number to yield single node, constant correction)
                      'hard_threshold_m': 10, # m
                      'pos_err_ref_frame': 'ned', # ['ecef' or 'ned'] The ref frame to estimate position errors in
                      'time_interpolation_method': 'linear',
                      'sigma_param' : np.array([2, 2, 5, 0.1, 0.1, 1]) # north [m], east [m], down [m], roll [deg], pitch [deg], yaw [deg] (is different for RTK/PPK!!!!)
                      }
    else:
        # When no coregistration is done, then resample datacube
        custom_config['Orthorectification']['resample_rgb_only'] = False
    
    # 
    if fast_mode:
        custom_config['Orthorectification']['resample_rgb_only'] = False
        custom_config['Orthorectification']['resolutionhyperspectralmosaic'] = 1


    # Customizes the config file according to settings
    customize_config(config_path=config_file_mission, dict_custom=custom_config)


    config = configparser.ConfigParser()
    config.read(config_file_mission)
    return config, config_specim_preprocess, config_file_mission


In [2]:

specim_mission_folder = "/home/notebook/cogs_massimal/massimal_bodo_saltstraumen_202203121143-small_hsi"
config_yaml = os.path.join(specim_mission_folder, "config.seabee.yaml")
geoid_path = os.path.join("/home/notebook/cogs_massimal", "no_kv_HREF2018A_NN2000_EUREF89.tif")
config_template_path = os.path.join("/home/notebook/cogs_massimal", "config_template_path_specim.ini")
lab_calibration_path = ''
afov = np.deg2rad(36.5) # degrees

# Choose the product that you want to georeference
processing_lvl  = "2b" # 
preprocess_lvl_dict = {"0": "0_raw",
                       "1a": "1a_radiance",
                       "2a": "2a_reflectance",
                       "2b": "2b_reflectance_gc"} # key word, subfolder name





config, config_specim_preprocess, config_file = define_processing_vars(config_yaml, 
                       specim_mission_folder, 
                       geoid_path, 
                       config_template_path, 
                       lab_calibration_path, 
                       fast_mode = True)

DEM folder does not exist so Geoid is used as terrain instead
Coregistration is not done, as there was no reference orthomosaic


# 1 Locate the corrected data cubes

In [3]:
import glob
# Patterns for searching data cube files
PATTERN_ENVI = '*.hdr'

# Expects the captured data to reside in capture subfolder
capture_dir = os.path.join(specim_mission_folder,
                           "pre-processed",
                           preprocess_lvl_dict[processing_lvl])

# Path for searching the ENVI files
search_path_envi = os.path.normpath(os.path.join(capture_dir, PATTERN_ENVI))
# Finding all files
envi_hdr_file_paths = glob.glob(search_path_envi)
len(envi_hdr_file_paths)

22

In [4]:
from spectral import envi
import pymap3d as pm
import pyproj
import rasterio



# Helper function
def get_geoid_undulation(src, latitude, longitude):
    """Extracts geoid undulation from a GeoTIFF at a given point.

    Args:
        src: Rasterio dataset object.
        latitude: Latitude in decimal degrees.
        longitude: Longitude in decimal degrees.

    Returns:
        Geoid undulation in meters.
    """

    # Transform coordinates to raster pixel coordinates
    row, col = src.index(longitude, latitude)

    # Extract pixel value (geoid undulation)
    geoid_undulation = src.read(1)[row, col]

    return geoid_undulation
# Define metadata
# Read all meta data from header file (currently hard coded, but could be avoided I guess)
# An instance of ResononImage will be created for each image
class ResononImage:
    def __init__(self, envi_hdr_file_path, config):
        """Initialize with an envi HDR"""
        self.spectral_image_obj = envi.open(envi_hdr_file_path)
        # Verbosely written out 
        self.n_lines = int(self.spectral_image_obj.metadata['lines'])
        self.n_bands = int(self.spectral_image_obj.metadata['bands'])
        self.n_pix = int(self.spectral_image_obj.metadata['samples'])
        self.binning_spatial = int(self.spectral_image_obj.metadata['sample binning'])
        self.binning_spectral = int(self.spectral_image_obj.metadata['spectral binning'])
        
        self.file_type = self.spectral_image_obj.metadata['file type']
        self.hdr_offset = int(self.spectral_image_obj.metadata['header offset'])
        self.interleave = self.spectral_image_obj.metadata['interleave']
        self.byte_order = self.spectral_image_obj.metadata['byte order']
        self.t_exp_ms = float(self.spectral_image_obj.metadata['shutter'])
        self.t_exp_unit = self.spectral_image_obj.metadata['shutter units']
        self.fps = float(self.spectral_image_obj.metadata['framerate'])
        self.wlen_unit = self.spectral_image_obj.metadata['wavelength units']
        self.gain = float(self.spectral_image_obj.metadata['gain'])

        self.direction = self.spectral_image_obj.metadata['direction']
        self.flip_radiometric_calibration = self.spectral_image_obj.metadata['flip radiometric calibration']
        self.timestamp = self.spectral_image_obj.metadata['timestamp'] # A single timestamp 

        self.target = self.spectral_image_obj.metadata['target']
        self.wavelengths = self.spectral_image_obj.metadata['wavelength']
        
        self.datacube = self.spectral_image_obj[:,:,:] # Load the datacube
    
# The processing in massipipe nicely renders the necessary navigation data in a json format
    def process_nav_json(self, json_file, geoid_path):
        """Reads"""
        with open(json_file, 'r') as f:
            data = json.load(f)
        # Unravel data into variables
        lat = np.array(data['latitude'])
        lon = np.array(data['longitude'])
        
        # Allow sampling of geoid height
        with rasterio.open(geoid_path, 'r') as src:
            geoid_height = get_geoid_undulation(src, lat, lon)
        
        alt_msl = np.array(data['altitude']) # Is above geoid
        
        alt_ell = alt_msl + geoid_height.reshape(alt_msl.shape) # above ellipsoid
        
        roll = np.array(data['roll'])
        pitch = np.array(data['pitch'])
        yaw = np.array(data['yaw'])
        
        timestamp = np.array(data['time']).reshape((-1, 1)) # Only relative time
        
        # If your nav system gave you geodetic positions, convert them to earth centered earth fixed (ECEF). Make sure to use ellipsoid height (not height above mean sea level (MSL) aka geoid)
        x, y, z = pm.geodetic2ecef(lat = lat, lon = lon, alt = alt_ell, deg=True)

        # Roll pitch yaw are ordered with in an unintuitive attribute name eul_zyx. The euler angles with rotation order ZYX are Yaw Pitch Roll
        self.eul_zyx = np.vstack((roll, pitch, yaw)).T

        # The ECEF positions
        self.position_ecef = np.vstack((x,y,z)).T
        
        print(self.eul_zyx.shape)
        print(self.position_ecef.shape)

        self.nav_timestamp = timestamp
        self.hsi_timestamps = timestamp

template_img = ResononImage(envi_hdr_file_paths[-1], config)



# 2. Describe the fov of the camera (in terms of a camera model) and the rotation/translation of the camera wrt IMU

In [11]:
from scipy.spatial.transform import Rotation as RotLib
from gref4hsi.utils.geometry_utils import CalibHSI

if lab_calibration_path == '':
    # This means that there is no manufacturer precise calibration for the FOV. Assume a pinhole model:
    # See https://github.com/havardlovas/gref4hsi for info
    width = template_img.n_pix
    cx = width/2
    f = width / (2*np.tan(afov/2))
    k1, k2, k3 = 0, 0, 0
    
else:
    pass

# We define the rotations/translations from the user input:

# User set rotation matrix
R_hsi_body = config_specim_preprocess.rotation_matrix_hsi_to_body

r_zyx = RotLib.from_matrix(R_hsi_body).as_euler('ZYX', degrees=False)

# Euler angle representation (any other would do too)
rotation_z = r_zyx[0]
rotation_y = r_zyx[1]
rotation_x = r_zyx[2]

# Vector from origin of HSI to body origin, expressed in body
# User set
t_hsi_body = config_specim_preprocess.translation_body_to_hsi
translation_x = t_hsi_body[0]
translation_y = t_hsi_body[1]
translation_z = t_hsi_body[2]



param_dict = {'rx':rotation_x,
              'ry':rotation_y,
              'rz':rotation_z,
              'tx':translation_x,
              'ty':translation_y,
              'tz':translation_z,
              'f': f,
              'cx': cx,
              'k1': k1,
              'k2': k2,
              'k3': k3,
              'width': width}

file_name_xml = 'HSI_' + str(template_img.binning_spatial) + 'b.xml'



camera_calib_xml_dir = config['Absolute Paths']['calib_folder'] # Where we put geometric calib files

xml_cal_write_path = os.path.join(camera_calib_xml_dir, file_name_xml)

CalibHSI(file_name_cal_xml= xml_cal_write_path, 
                    mode = 'w', 
                    param_dict = param_dict)

# Set value in config file:
config.set('Absolute Paths', 'hsi_calib_path', value = xml_cal_write_path)
# Write the config object 
with open(config_file, 'w') as configfile:
        config.write(configfile)

# 3. Extraction of navigation data

In [6]:

    # 

import json
nav_pattern = '*.json'

nav_dir = os.path.join(specim_mission_folder,
                           "pre-processed",
                           'imudata')
search_path_nav = os.path.normpath(os.path.join(nav_dir, nav_pattern))
nav_file_paths = glob.glob(search_path_nav)


# Let's try to find the 9th image data
search_path_nav = os.path.normpath(os.path.join(nav_dir, "*001*"))
nav_file_path = glob.glob(search_path_nav)
print(nav_file_path)



['/home/notebook/cogs_massimal/massimal_bodo_saltstraumen_202203121143-small_hsi/pre-processed/imudata/pre-processed_001_imudata.json']


In [7]:
import h5py

# Defining the mapping from the ResononImage object to the h5 file
h5_dict_write = {'eul_zyx' : config['HDF.raw_nav']['eul_zyx'],
            'position_ecef' : config['HDF.raw_nav']['position'],
            'nav_timestamp' : config['HDF.raw_nav']['timestamp'],
            'datacube': config['HDF.hyperspectral']['datacube'],
            't_exp_ms': config['HDF.hyperspectral']['exposuretime'],
            'hsi_timestamps': config['HDF.hyperspectral']['timestamp'],
            'wavelengths' : config['HDF.calibration']['band2wavelength']}

# Defining a writer for the relevant attributes
def resonon_object_2_h5_file(h5_filename, h5_tree_dict, resonon_object):
    with h5py.File(h5_filename, 'w', libver='latest') as f:
        for attribute_name, h5_hierarchy_item_path in h5_tree_dict.items():
            dset = f.create_dataset(name=h5_hierarchy_item_path, 
                                            data = getattr(resonon_object, attribute_name)) 


In [8]:

# For legacy reasons the software expects data to come in the h5 file format. 
# TODO: change that
h5_folder = config['Absolute Paths']['h5_folder']

for i in range(len(nav_file_paths)):
    img_id = f"*{i:03d}*"
    transect_nr = f"{i:03d}"
    
    search_path_nav = os.path.normpath(os.path.join(nav_dir, img_id + '.json'))
    search_path_envi_hdr = os.path.normpath(os.path.join(capture_dir, img_id + '.hdr'))
    
    envi_hdr_file = glob.glob(search_path_envi_hdr)[0]
    nav_file = glob.glob(search_path_nav)[0]
    
    reson_img = ResononImage(envi_hdr_file, config)
    reson_img.process_nav_json(nav_file, geoid_path)
    
    transect_name = os.path.basename(envi_hdr_file).split(sep = '.')[0] # To remove suffix
    
    # Possible to name files with <PREFIX>_<time_start>_<Transect#>_<Chunk#>.h5
    h5_filename = h5_folder + transect_name + '.h5'

    resonon_object_2_h5_file(h5_filename=h5_filename, 
                             h5_tree_dict=h5_dict_write, 
                             resonon_object=reson_img)
    
    print(f"Image nr {i:03d}")

(2000, 3)
(2000, 3)
[[0.0000000e+00]
 [8.3330000e-03]
 [1.6667000e-02]
 ...
 [1.6641685e+01]
 [1.6650019e+01]
 [1.6655000e+01]]
Image nr 000
(179, 3)
(179, 3)
[[0.        ]
 [0.008333  ]
 [0.016667  ]
 [0.025     ]
 [0.033333  ]
 [0.041667  ]
 [0.05      ]
 [0.058333  ]
 [0.066667  ]
 [0.075     ]
 [0.083333  ]
 [0.091667  ]
 [0.1       ]
 [0.108333  ]
 [0.116667  ]
 [0.125     ]
 [0.133333  ]
 [0.141667  ]
 [0.15      ]
 [0.158333  ]
 [0.166667  ]
 [0.175     ]
 [0.183333  ]
 [0.191667  ]
 [0.2       ]
 [0.208333  ]
 [0.216667  ]
 [0.225     ]
 [0.233333  ]
 [0.241667  ]
 [0.25      ]
 [0.258333  ]
 [0.266667  ]
 [0.275     ]
 [0.283334  ]
 [0.291667  ]
 [0.3       ]
 [0.308334  ]
 [0.316667  ]
 [0.325     ]
 [0.333334  ]
 [0.341667  ]
 [0.35      ]
 [0.358334  ]
 [0.366667  ]
 [0.375     ]
 [0.383334  ]
 [0.391667  ]
 [0.4       ]
 [0.408334  ]
 [0.416667  ]
 [0.425     ]
 [0.433334  ]
 [0.441667  ]
 [0.45      ]
 [0.458334  ]
 [0.466667  ]
 [0.475     ]
 [0.483334  ]
 [0.491667  ]
 

In [9]:
from gref4hsi.scripts import georeference, orthorectification, coregistration
from gref4hsi.utils import parsing_utils, specim_parsing_utils

# Time interpolates and reformats the pose (of the vehicle body) to "processed/nav/" folder.
parsing_utils.export_pose(config_file)

# Formats model to triangular mesh and an earth centered earth fixed / geocentric coordinate system
parsing_utils.export_model(config_file)

# Commenting out the georeference step is fine if it has been done


## Visualize the data 3D photo model from RGB images and the time-resolved positions/orientations
#visualize.show_mesh_camera(config, show_mesh = True, show_pose = True, ref_frame='ENU')

# Step 1: Direct georeferencing
georeference.main(config_file)

# Step 2: Orthorectify the direct georeferenced data (incl metadata)
orthorectification.main(config_file)

Computing 2D Triangulation: 100%|██████████[00:00<00:00]



################ Georeferencing with a Brake: ################
pre-processed_000_reflectance_gc.h5
Georeferencing file 1/22, progress is 0.0 %


FileNotFoundError: [Errno 2] No such file or directory: '/home/notebook/cogs_massimal/massimal_bodo_saltstraumen_202203121143-small_hsi/processed/Input/Calib/HSI_2b.xml'

In [ ]:
%debug